# 📉 SPC Module — Statistical Process Control for Wafer Probing
### Add-on to: PTSD Wafer Probing Simulation (IEEE ITC 2023)

---
**What is SPC?**  
Statistical Process Control (SPC) uses control charts to monitor a process over time and detect when it goes **out of control** — before failures occur. In yield engineering, SPC on CRES data tells you *exactly when* to clean the probe, rather than guessing.

**Charts in this module:**
- 📊 **X-bar Chart** — Monitors average CRES per wafer batch
- 📊 **R Chart** — Monitors CRES variability (range)
- 📊 **CUSUM Chart** — Detects small, sustained shifts in CRES (most sensitive)
- 📊 **Western Electric Rules** — Flags subtle out-of-control patterns
- 📊 **Cp / Cpk** — Process capability indices (key interview metric!)
---

In [ ]:
!pip install -q numpy pandas matplotlib seaborn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import os
os.makedirs('outputs', exist_ok=True)
np.random.seed(42)

print('✅ Libraries loaded!')

## ⚙️ Step 1 — Generate / Load CRES Sensor Data

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================
N_WAFERS      = 100    # Number of wafer batches to monitor
SUBGROUP_SIZE = 5      # Measurements per wafer (n=5 is industry standard)
UCL_SIGMA     = 3      # Control limit: 3-sigma (99.73% of normal process)

# CRES specification limits (from paper context)
USL = 1.5   # Upper Specification Limit (Ohm) — failure threshold from paper
LSL = 0.05  # Lower Specification Limit (Ohm)
TARGET = 0.5  # Target CRES value (Ohm)

# ================================================================
# SIMULATE CRES DATA OVER TIME
# Mimics real probe degradation: stable → gradual drift → spike
# ================================================================

# Phase 1 (wafers 1-50):   Clean probe, stable CRES
# Phase 2 (wafers 51-75):  Gradual contamination buildup
# Phase 3 (wafers 76-100): Heavy contamination / bonded debris

cres_data = []
for w in range(N_WAFERS):
    if w < 50:
        mean = 0.50; std = 0.05          # Stable
    elif w < 75:
        mean = 0.50 + (w-50)*0.015; std = 0.08   # Gradual drift
    else:
        mean = 0.875 + (w-75)*0.03; std = 0.15   # Heavy contamination
    samples = np.random.normal(mean, std, SUBGROUP_SIZE)
    samples = np.clip(samples, 0.05, 3.0)
    cres_data.append(samples)

cres_data = np.array(cres_data)  # Shape: (N_WAFERS, SUBGROUP_SIZE)

# Subgroup statistics
xbar = cres_data.mean(axis=1)   # Mean per wafer
R    = cres_data.max(axis=1) - cres_data.min(axis=1)  # Range per wafer

print(f'✅ CRES data generated: {N_WAFERS} wafers × {SUBGROUP_SIZE} measurements')
print(f'   Overall mean CRES: {xbar.mean():.4f} Ω')
print(f'   Overall CRES std:  {cres_data.std():.4f} Ω')

## 📊 Step 2 — X-bar & R Control Charts

In [ ]:
# ================================================================
# X-BAR & R CHART
# Industry standard SPC charts for subgroup data
# Control limits based on first 25 wafers (baseline / "in-control" phase)
# ================================================================

# SPC constants for n=5 (from standard SPC tables)
A2 = 0.577   # For X-bar chart control limits
D3 = 0.0     # Lower control limit factor for R chart
D4 = 2.114   # Upper control limit factor for R chart

# Baseline: use first 25 wafers (stable phase)
BASELINE = 25
xbar_baseline = xbar[:BASELINE].mean()
R_baseline    = R[:BASELINE].mean()

# Control limits
xbar_UCL = xbar_baseline + A2 * R_baseline
xbar_LCL = xbar_baseline - A2 * R_baseline
R_UCL    = D4 * R_baseline
R_LCL    = D3 * R_baseline

# Detect out-of-control points
xbar_ooc = (xbar > xbar_UCL) | (xbar < xbar_LCL)
R_ooc    = (R > R_UCL) | (R < R_LCL)

wafer_ids = np.arange(1, N_WAFERS+1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('X-bar & R Control Chart — CRES Monitoring\n(Wafer Probe Needle Contamination Tracking)',
             fontsize=13, fontweight='bold')

# --- X-bar Chart ---
ax1.plot(wafer_ids, xbar, 'b-o', markersize=4, linewidth=1.5, label='Subgroup Mean (X-bar)')
ax1.axhline(xbar_UCL, color='red',   linestyle='--', linewidth=1.5, label=f'UCL = {xbar_UCL:.3f} Ω')
ax1.axhline(xbar_LCL, color='red',   linestyle='--', linewidth=1.5, label=f'LCL = {xbar_LCL:.3f} Ω')
ax1.axhline(xbar_baseline, color='green', linestyle='-', linewidth=1.5, label=f'CL = {xbar_baseline:.3f} Ω')
ax1.axhline(USL, color='purple', linestyle=':', linewidth=1.5, label=f'USL = {USL} Ω (paper threshold)')
ax1.scatter(wafer_ids[xbar_ooc], xbar[xbar_ooc], color='red', s=80, zorder=5, label='Out-of-Control ⚠️')
ax1.axvspan(50, 75, alpha=0.08, color='orange', label='Contamination Buildup Zone')
ax1.axvspan(75, 100, alpha=0.12, color='red', label='Heavy Contamination Zone')
ax1.set_ylabel('Mean CRES (Ω)', fontsize=11)
ax1.legend(fontsize=8, loc='upper left', ncol=2)
ax1.grid(True, alpha=0.3)
ax1.set_title('X-bar Chart (Process Mean)', fontsize=11)

# --- R Chart ---
ax2.plot(wafer_ids, R, 'b-o', markersize=4, linewidth=1.5, label='Subgroup Range (R)')
ax2.axhline(R_UCL, color='red',   linestyle='--', linewidth=1.5, label=f'UCL = {R_UCL:.3f} Ω')
ax2.axhline(R_LCL, color='red',   linestyle='--', linewidth=1.5, label=f'LCL = {R_LCL:.3f} Ω')
ax2.axhline(R_baseline, color='green', linestyle='-', linewidth=1.5, label=f'CL = {R_baseline:.3f} Ω')
ax2.scatter(wafer_ids[R_ooc], R[R_ooc], color='red', s=80, zorder=5, label='Out-of-Control ⚠️')
ax2.set_xlabel('Wafer Number', fontsize=11)
ax2.set_ylabel('Range of CRES (Ω)', fontsize=11)
ax2.legend(fontsize=8, loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_title('R Chart (Process Variability)', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/SPC_Fig1_Xbar_R_Chart.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 X-bar Chart Results:')
print(f'   Center Line (CL) : {xbar_baseline:.4f} Ω')
print(f'   Upper Control Limit (UCL): {xbar_UCL:.4f} Ω')
print(f'   Lower Control Limit (LCL): {xbar_LCL:.4f} Ω')
print(f'   Out-of-control points: {xbar_ooc.sum()} wafers ⚠️')
print(f'\n📊 R Chart Results:')
print(f'   Out-of-control points: {R_ooc.sum()} wafers ⚠️')

## 📊 Step 3 — CUSUM Chart (Most Sensitive to Drift)

In [ ]:
# ================================================================
# CUSUM CHART (Cumulative Sum)
# Detects small, sustained shifts that X-bar charts miss
# Critical for catching early contamination buildup
# ================================================================

# CUSUM parameters
mu0   = xbar_baseline          # In-control mean
sigma = (R_baseline / 2.326)   # Estimated sigma from R-bar / d2 (n=5)
K     = 0.5 * sigma            # Allowable slack (k=0.5 is standard)
H     = 5.0 * sigma            # Decision interval (h=5 is standard)

# Compute CUSUM (upper and lower)
cusum_pos = np.zeros(N_WAFERS)  # Detects upward shifts (rising CRES)
cusum_neg = np.zeros(N_WAFERS)  # Detects downward shifts
for i in range(1, N_WAFERS):
    cusum_pos[i] = max(0, cusum_pos[i-1] + (xbar[i] - mu0) - K)
    cusum_neg[i] = max(0, cusum_neg[i-1] - (xbar[i] - mu0) - K)

cusum_alarm = (cusum_pos > H) | (cusum_neg > H)
first_alarm = np.where(cusum_alarm)[0]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(wafer_ids, cusum_pos, 'b-o', markersize=4, linewidth=1.5, label='CUSUM+ (upward shift)')
ax.plot(wafer_ids, cusum_neg, 'o-', color='orange', markersize=4, linewidth=1.5, label='CUSUM- (downward shift)')
ax.axhline(H, color='red', linestyle='--', linewidth=2, label=f'Decision Interval H = {H:.3f} Ω')
ax.axhline(0, color='green', linestyle='-', linewidth=1)

if len(first_alarm) > 0:
    ax.axvline(first_alarm[0]+1, color='red', linestyle=':', linewidth=2,
               label=f'🚨 First Alarm: Wafer #{first_alarm[0]+1}')
    ax.scatter(wafer_ids[cusum_alarm], cusum_pos[cusum_alarm],
               color='red', s=60, zorder=5)

ax.axvspan(50, 75, alpha=0.08, color='orange')
ax.axvspan(75, 100, alpha=0.12, color='red')
ax.set_xlabel('Wafer Number', fontsize=11)
ax.set_ylabel('Cumulative Sum of CRES (Ω)', fontsize=11)
ax.set_title('CUSUM Control Chart — Early Contamination Detection\n'
             '(Detects gradual CRES drift before X-bar chart reacts)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/SPC_Fig2_CUSUM_Chart.png', dpi=150, bbox_inches='tight')
plt.show()

if len(first_alarm) > 0:
    print(f'🚨 CUSUM first alarm at Wafer #{first_alarm[0]+1}')
    print(f'   This is BEFORE the X-bar alarm — CUSUM detects drift earlier!')
print(f'   Total alarms: {cusum_alarm.sum()} wafers')
print(f'   Sigma estimate: {sigma:.4f} Ω | K={K:.4f} | H={H:.4f}')

## 📊 Step 4 — Western Electric Rules

In [ ]:
# ================================================================
# WESTERN ELECTRIC (WE) RULES
# Industry-standard pattern detection for X-bar charts
# These catch subtle out-of-control signals not visible from single points
# ================================================================

sigma_est = (R_baseline / 2.326)
z = (xbar - xbar_baseline) / sigma_est  # Standardized values

# Rule 1: Any point beyond 3-sigma (basic out-of-control)
rule1 = np.abs(z) > 3

# Rule 2: 2 out of 3 consecutive points beyond 2-sigma (same side)
rule2 = np.zeros(N_WAFERS, dtype=bool)
for i in range(2, N_WAFERS):
    window = z[i-2:i+1]
    if np.sum(window > 2) >= 2 or np.sum(window < -2) >= 2:
        rule2[i] = True

# Rule 3: 4 out of 5 consecutive points beyond 1-sigma (same side)
rule3 = np.zeros(N_WAFERS, dtype=bool)
for i in range(4, N_WAFERS):
    window = z[i-4:i+1]
    if np.sum(window > 1) >= 4 or np.sum(window < -1) >= 4:
        rule3[i] = True

# Rule 4: 8 consecutive points on same side of center line (trend)
rule4 = np.zeros(N_WAFERS, dtype=bool)
for i in range(7, N_WAFERS):
    window = z[i-7:i+1]
    if np.all(window > 0) or np.all(window < 0):
        rule4[i] = True

any_violation = rule1 | rule2 | rule3 | rule4

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(wafer_ids, xbar, 'b-o', markersize=3, linewidth=1.2, alpha=0.7, label='X-bar CRES')

# Control limit bands
for sigma_level, color, alpha in [(3,'red',0.08),(2,'orange',0.08),(1,'yellow',0.08)]:
    ax.axhspan(xbar_baseline + sigma_level*sigma_est,
               xbar_baseline + (sigma_level+1 if sigma_level<3 else 10)*sigma_est,
               alpha=alpha, color=color)
    ax.axhspan(xbar_baseline - sigma_level*sigma_est,
               xbar_baseline - (sigma_level+1 if sigma_level<3 else 10)*sigma_est,
               alpha=alpha, color=color)

for sigma_level, label in [(3,'±3σ (UCL/LCL)'),(2,'±2σ'),(1,'±1σ')]:
    ax.axhline(xbar_baseline + sigma_level*sigma_est, color='gray', linestyle='--', linewidth=0.8)
    ax.axhline(xbar_baseline - sigma_level*sigma_est, color='gray', linestyle='--', linewidth=0.8)
ax.axhline(xbar_baseline, color='green', linewidth=1.5, label='Center Line')

# Plot violations
colors_rules = {'Rule 1 (>3σ)': ('red', rule1),
                'Rule 2 (2/3 >2σ)': ('orange', rule2),
                'Rule 3 (4/5 >1σ)': ('purple', rule3),
                'Rule 4 (8 same side)': ('brown', rule4)}
markers = ['X','s','^','D']
for (label, (color, rule)), marker in zip(colors_rules.items(), markers):
    if rule.any():
        ax.scatter(wafer_ids[rule], xbar[rule], color=color, s=80,
                   zorder=5, marker=marker, label=f'⚠️ {label}: {rule.sum()} pts')

ax.set_xlabel('Wafer Number', fontsize=11)
ax.set_ylabel('Mean CRES (Ω)', fontsize=11)
ax.set_title('Western Electric Rules — Out-of-Control Pattern Detection\n'
             'CRES Monitoring for Probe Needle Contamination',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper left', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/SPC_Fig3_Western_Electric_Rules.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Western Electric Rule Violations:')
print(f'   Rule 1 (>3σ):           {rule1.sum():3d} violations')
print(f'   Rule 2 (2/3 beyond 2σ): {rule2.sum():3d} violations')
print(f'   Rule 3 (4/5 beyond 1σ): {rule3.sum():3d} violations')
print(f'   Rule 4 (8 same side):   {rule4.sum():3d} violations')
print(f'   Total unique violations: {any_violation.sum()} wafers')

## 📊 Step 5 — Process Capability (Cp & Cpk)

In [ ]:
# ================================================================
# PROCESS CAPABILITY INDICES — Cp and Cpk
# KEY metric in yield engineering interviews!
# Cp  = (USL - LSL) / (6σ)          — process spread vs spec
# Cpk = min(CPU, CPL)                — accounts for centering
# Target: Cp & Cpk > 1.33 (4-sigma) or > 1.67 (5-sigma) for semiconductors
# ================================================================

# Compute for each phase
phases = {
    'Phase 1 (Clean — Wafers 1-50)':       cres_data[:50].flatten(),
    'Phase 2 (Drift — Wafers 51-75)':      cres_data[50:75].flatten(),
    'Phase 3 (Contaminated — Wafers 76-100)': cres_data[75:].flatten()
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Process Capability Analysis (Cp & Cpk) — CRES per Phase',
             fontsize=13, fontweight='bold')

results_cpk = []
for ax, (phase_name, data) in zip(axes, phases.items()):
    mu    = data.mean()
    sigma = data.std()
    Cp    = (USL - LSL) / (6 * sigma)
    CPU   = (USL - mu)  / (3 * sigma)
    CPL   = (mu - LSL)  / (3 * sigma)
    Cpk   = min(CPU, CPL)

    results_cpk.append({'Phase': phase_name, 'Mean': mu, 'Sigma': sigma,
                        'Cp': Cp, 'Cpk': Cpk})

    x = np.linspace(LSL - 0.2, USL + 0.5, 300)
    pdf = stats.norm.pdf(x, mu, sigma)

    color = 'steelblue' if Cpk >= 1.33 else ('orange' if Cpk >= 1.0 else 'crimson')
    ax.plot(x, pdf, color=color, linewidth=2.5)
    ax.fill_between(x, pdf, where=(x >= LSL) & (x <= USL), alpha=0.3, color=color)
    ax.fill_between(x, pdf, where=(x > USL), alpha=0.5, color='red', label='Out-of-spec')
    ax.fill_between(x, pdf, where=(x < LSL), alpha=0.5, color='red')
    ax.axvline(USL, color='red', linestyle='--', linewidth=2, label=f'USL={USL}Ω')
    ax.axvline(LSL, color='red', linestyle='--', linewidth=2, label=f'LSL={LSL}Ω')
    ax.axvline(mu, color=color, linestyle='-', linewidth=2, label=f'Mean={mu:.3f}Ω')
    ax.axvline(TARGET, color='green', linestyle=':', linewidth=1.5, label=f'Target={TARGET}Ω')

    status = '✅ Capable' if Cpk >= 1.33 else ('⚠️ Marginal' if Cpk >= 1.0 else '❌ Incapable')
    ax.set_title(f'{phase_name[:25]}\nCp={Cp:.2f} | Cpk={Cpk:.2f} | {status}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('CRES (Ω)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/SPC_Fig4_Process_Capability.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Process Capability Summary:')
print(f'{"Phase":<45} {"Mean":>7} {"Sigma":>7} {"Cp":>6} {"Cpk":>6} {"Status"}')
print('-'*85)
for r in results_cpk:
    status = '✅ Capable' if r['Cpk'] >= 1.33 else ('⚠️ Marginal' if r['Cpk'] >= 1.0 else '❌ Incapable')
    print(f"{r['Phase']:<45} {r['Mean']:>7.4f} {r['Sigma']:>7.4f} {r['Cp']:>6.2f} {r['Cpk']:>6.2f}  {status}")
print('\nNote: Semiconductor industry target: Cpk > 1.33 (4-sigma capability)')

## 📋 Step 6 — SPC Summary Dashboard

In [ ]:
# ================================================================
# FULL SPC SUMMARY DASHBOARD
# Interview-ready single figure summarizing all SPC findings
# ================================================================

fig = plt.figure(figsize=(16, 10))
fig.suptitle('SPC Dashboard — Wafer Probe CRES Monitoring\nPTSD Framework | IEEE ITC 2023',
             fontsize=14, fontweight='bold', y=0.98)

# Top left: X-bar
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(wafer_ids, xbar, 'b-', linewidth=1.2)
ax1.axhline(xbar_UCL, color='red', linestyle='--', linewidth=1.5, label=f'UCL={xbar_UCL:.3f}')
ax1.axhline(xbar_LCL, color='red', linestyle='--', linewidth=1.5, label=f'LCL={xbar_LCL:.3f}')
ax1.axhline(xbar_baseline, color='green', linewidth=1.5, label=f'CL={xbar_baseline:.3f}')
ax1.scatter(wafer_ids[xbar_ooc], xbar[xbar_ooc], color='red', s=50, zorder=5)
ax1.set_title('X-bar Chart', fontweight='bold'); ax1.set_ylabel('CRES (Ω)')
ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)

# Top right: CUSUM
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(wafer_ids, cusum_pos, 'b-', linewidth=1.2, label='CUSUM+')
ax2.axhline(H, color='red', linestyle='--', linewidth=1.5, label=f'H={H:.3f}')
ax2.scatter(wafer_ids[cusum_alarm], cusum_pos[cusum_alarm], color='red', s=50, zorder=5)
ax2.set_title('CUSUM Chart', fontweight='bold'); ax2.set_ylabel('Cumulative Sum')
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)

# Bottom left: Violations bar
ax3 = fig.add_subplot(2, 2, 3)
rules_counts = [rule1.sum(), rule2.sum(), rule3.sum(), rule4.sum()]
rules_labels = ['Rule 1\n(>3σ)', 'Rule 2\n(2/3>2σ)', 'Rule 3\n(4/5>1σ)', 'Rule 4\n(8 same)']
colors_bar   = ['red' if c > 0 else 'steelblue' for c in rules_counts]
bars = ax3.bar(rules_labels, rules_counts, color=colors_bar, edgecolor='white')
ax3.bar_label(bars, fontsize=11, fontweight='bold')
ax3.set_title('Western Electric Rule Violations', fontweight='bold')
ax3.set_ylabel('Number of Violations'); ax3.grid(axis='y', alpha=0.3)

# Bottom right: Cpk trend
ax4 = fig.add_subplot(2, 2, 4)
cpk_vals   = [r['Cpk'] for r in results_cpk]
phase_lbls = ['Phase 1\n(Clean)', 'Phase 2\n(Drift)', 'Phase 3\n(Contam.)']
bar_colors = ['green' if c >= 1.33 else ('orange' if c >= 1.0 else 'red') for c in cpk_vals]
bars2 = ax4.bar(phase_lbls, cpk_vals, color=bar_colors, edgecolor='white')
ax4.bar_label(bars2, fmt='%.2f', fontsize=11, fontweight='bold')
ax4.axhline(1.33, color='green', linestyle='--', linewidth=1.5, label='Target Cpk=1.33')
ax4.axhline(1.00, color='orange', linestyle='--', linewidth=1.5, label='Min Cpk=1.00')
ax4.set_title('Process Capability (Cpk) by Phase', fontweight='bold')
ax4.set_ylabel('Cpk Value'); ax4.legend(fontsize=8); ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/SPC_Fig5_Dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ SPC Module Complete! Files saved to outputs/')
print('   SPC_Fig1_Xbar_R_Chart.png')
print('   SPC_Fig2_CUSUM_Chart.png')
print('   SPC_Fig3_Western_Electric_Rules.png')
print('   SPC_Fig4_Process_Capability.png')
print('   SPC_Fig5_Dashboard.png')
print('\n🎯 Interview Tip: Be ready to explain:')
print('   → Why CUSUM detects drift earlier than X-bar')
print('   → What Cpk > 1.33 means for yield')
print('   → How SPC triggers the cleaning cycle in the PTSD framework')

In [ ]:
# Download all outputs (Colab only)
try:
    from google.colab import files
    import zipfile
    with zipfile.ZipFile('SPC_outputs.zip', 'w') as zf:
        for f in os.listdir('outputs'):
            if f.startswith('SPC'):
                zf.write(f'outputs/{f}', f)
    files.download('SPC_outputs.zip')
    print('✅ Download started!')
except ImportError:
    print('ℹ️  Files are in outputs/ folder.')